# Tabular Pipeline and Model Screening

Builds the 929-feature applicant-level table and screens the tabular baseline models.

This is the publication copy for the SAGE-Credit repository. The original analysis logic is preserved, machine-specific paths are replaced with portable configuration, and cached outputs are cleared.

> Download the Home Credit Default Risk competition files separately; the dataset is not redistributed here.

In [ ]:
from pathlib import Path
import os

# Portable repository configuration. Override either location with an environment variable.
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
DATA_DIR = Path(os.environ.get("HOME_CREDIT_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser().resolve()
ARTIFACT_DIR = Path(os.environ.get("SAGE_CREDIT_ARTIFACT_DIR", PROJECT_ROOT / "artifacts")).expanduser().resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {DATA_DIR}")
print(f"Artifact directory: {ARTIFACT_DIR}")


In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

application_train = pd.read_csv(DATA_DIR / "application_train.csv")

def clean():
  gc.collect()

def safeDivide(numerator, denominator):
    result = numerator / denominator.replace(0, np.nan)
    result = result.replace([np.inf, -np.inf], np.nan)
    return result

def astypeFloat32(df):
  for col in df.columns:
    if col != 'SK_ID_CURR' and col != 'TARGET':
      if df[col].dtype != 'object':
        df[col] = df[col].astype(np.float32)
  return df

def mergeFeatureBlock(master, features, name):
  print('Before ' + name + ':', master.shape)
  master = master.merge(features, on='SK_ID_CURR', how='left')
  print('After ' + name + ':', master.shape)
  return master

def checkFeatureBlock(df, name):
  print(name)
  print('Shape:', df.shape)
  print('Duplicate SK_ID_CURR:', df['SK_ID_CURR'].duplicated().sum())
  print('Missing SK_ID_CURR:', df['SK_ID_CURR'].isna().sum())
  print('Object columns:', df.select_dtypes(include=['object']).columns.tolist())
  print('Int columns:', df.select_dtypes(include=['int', 'int64', 'int32']).columns.tolist()[:10])
  print('Float columns:', df.select_dtypes(include=['float', 'float64', 'float32']).columns.tolist()[:10])
  print('Total missing values:', df.isna().sum().sum())

  plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 140,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25
})

In [ ]:
def buildApplicationFeatures(application_train):

    app = application_train.copy()

    app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)

    app['APP_CREDIT_INCOME_RATIO'] = safeDivide(app['AMT_CREDIT'], app['AMT_INCOME_TOTAL'])
    app['APP_ANNUITY_INCOME_RATIO'] = safeDivide(app['AMT_ANNUITY'], app['AMT_INCOME_TOTAL'])
    app['APP_ANNUITY_CREDIT_RATIO'] = safeDivide(app['AMT_ANNUITY'], app['AMT_CREDIT'])
    app['APP_GOODS_CREDIT_RATIO'] = safeDivide(app['AMT_GOODS_PRICE'], app['AMT_CREDIT'])
    app['APP_INCOME_PER_PERSON'] = safeDivide(app['AMT_INCOME_TOTAL'], app['CNT_FAM_MEMBERS'])
    app['APP_CREDIT_PER_PERSON'] = safeDivide(app['AMT_CREDIT'], app['CNT_FAM_MEMBERS'])

    ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

    app['APP_EXT_SOURCE_MEAN'] = app[ext_cols].mean(axis=1)
    app['APP_EXT_SOURCE_MIN'] = app[ext_cols].min(axis=1)
    app['APP_EXT_SOURCE_MAX'] = app[ext_cols].max(axis=1)
    app['APP_EXT_SOURCE_STD'] = app[ext_cols].std(axis=1)

    app = pd.get_dummies(app)

    for col in app.columns:
        if col not in ['SK_ID_CURR', 'TARGET']:
            if app[col].dtype != 'object':
                app[col] = app[col].fillna(app[col].median())
                app[col] = app[col].astype(np.float32)

    clean()

    return app

In [ ]:
credit_card_balance = pd.read_csv(DATA_DIR / "credit_card_balance.csv")

def buildCCFeatures(credit_card_balance):
  cc = credit_card_balance.copy()

  cc_overall = cc.groupby('SK_ID_CURR').agg(

      NUM_ACCOUNTS=('SK_ID_PREV', 'nunique'),

      BALANCE_MEAN=('AMT_BALANCE', 'mean'),
      BALANCE_MAX=('AMT_BALANCE', 'max'),
      BALANCE_MIN=('AMT_BALANCE', 'min'),
      BALANCE_STD=('AMT_BALANCE', 'std'),
      BALANCE_SUM=('AMT_BALANCE', 'sum'),

      CREDIT_LIMIT_MEAN=('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
      CREDIT_LIMIT_MAX=('AMT_CREDIT_LIMIT_ACTUAL', 'max'),

      DRAWINGS_MEAN=('AMT_DRAWINGS_CURRENT', 'mean'),
      DRAWINGS_MAX=('AMT_DRAWINGS_CURRENT', 'max'),
      DRAWINGS_SUM=('AMT_DRAWINGS_CURRENT', 'sum'),

      DRAWINGS_ATM_MEAN=('AMT_DRAWINGS_ATM_CURRENT', 'mean'),
      DRAWINGS_ATM_SUM=('AMT_DRAWINGS_ATM_CURRENT', 'sum'),

      DRAWINGS_POS_MEAN=('AMT_DRAWINGS_POS_CURRENT', 'mean'),
      DRAWINGS_POS_SUM=('AMT_DRAWINGS_POS_CURRENT', 'sum'),

      PAYMENT_MEAN=('AMT_PAYMENT_CURRENT', 'mean'),
      PAYMENT_MAX=('AMT_PAYMENT_CURRENT', 'max'),
      PAYMENT_SUM=('AMT_PAYMENT_CURRENT', 'sum'),

      PAYMENT_TOTAL_MEAN=('AMT_PAYMENT_TOTAL_CURRENT', 'mean'),
      PAYMENT_TOTAL_SUM=('AMT_PAYMENT_TOTAL_CURRENT', 'sum'),

      RECEIVABLE_MEAN=('AMT_TOTAL_RECEIVABLE', 'mean'),
      RECEIVABLE_MAX=('AMT_TOTAL_RECEIVABLE', 'max'),

      DRAWING_COUNT_MEAN=('CNT_DRAWINGS_CURRENT', 'mean'),
      DRAWING_COUNT_MAX=('CNT_DRAWINGS_CURRENT', 'max'),
      DRAWING_COUNT_SUM=('CNT_DRAWINGS_CURRENT', 'sum'),

      MAX_MATURE_INSTALLMENTS=('CNT_INSTALMENT_MATURE_CUM', 'max'),

      DPD_MEAN=('SK_DPD', 'mean'),
      DPD_MAX=('SK_DPD', 'max'),
      DPD_SUM=('SK_DPD', 'sum'),

      DPD_DEF_MEAN=('SK_DPD_DEF', 'mean'),
      DPD_DEF_MAX=('SK_DPD_DEF', 'max'),
      DPD_DEF_SUM=('SK_DPD_DEF', 'sum')

  ).reset_index()

  cc_cols = []
  for col in cc_overall.columns:
      if col == 'SK_ID_CURR':
          cc_cols.append(col)
      else:
          cc_cols.append('CC_OVERALL_' + col)
  cc_overall.columns = cc_cols

  for col in cc_overall.columns:

      if col != 'SK_ID_CURR':
          cc_overall[col] = cc_overall[col].astype(np.float32)

  del cc
  clean()

  return cc_overall

def buildCCWindowFeatures(credit_card_balance):

  cc = credit_card_balance

  windows = [1,3,6,12,24]

  cc_windows = None

  for window in windows:
    temp = cc[cc['MONTHS_BALANCE']>= -window]
    temp = temp.groupby('SK_ID_CURR').agg(
      BALANCE_MEAN=('AMT_BALANCE', 'mean'),
      BALANCE_MAX=('AMT_BALANCE', 'max'),

      CREDIT_LIMIT_MEAN=('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),

      DRAWINGS_MEAN=('AMT_DRAWINGS_CURRENT', 'mean'),
      DRAWINGS_SUM=('AMT_DRAWINGS_CURRENT', 'sum'),

      PAYMENT_MEAN=('AMT_PAYMENT_CURRENT', 'mean'),
      PAYMENT_SUM=('AMT_PAYMENT_CURRENT', 'sum'),

      RECEIVABLE_MEAN=('AMT_TOTAL_RECEIVABLE', 'mean'),

      DPD_MEAN=('SK_DPD', 'mean'),
      DPD_MAX=('SK_DPD', 'max'),

      DPD_DEF_MEAN=('SK_DPD_DEF', 'mean'),
      DPD_DEF_MAX=('SK_DPD_DEF', 'max'),

      ACTIVE_MONTHS=('MONTHS_BALANCE', 'count')
    ).reset_index()

    cols = []

    for col in temp.columns:
      if col == 'SK_ID_CURR':
        cols.append(col)
      else:
        cols.append('CC_WINDOW_' + str(window) + '_' + col)

    temp.columns = cols

    for col in cols:
      if col != 'SK_ID_CURR':
        temp[col] = temp[col].astype(np.float32)

    if cc_windows is None:
      cc_windows = temp
    else:
      cc_windows = cc_windows.merge(temp, on='SK_ID_CURR', how='left')

  del cc, temp
  clean()

  return cc_windows

def ccTrendFeatures(cc_features):
  cc = cc_features

  for window in [1, 3, 6, 12, 24]:
      if 'CC_WINDOW_' + str(window) + '_BALANCE_MEAN' in cc.columns:
          cc['CC_TREND_BALANCE_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
              cc['CC_WINDOW_' + str(window) + '_BALANCE_MEAN'], cc['CC_OVERALL_BALANCE_MEAN']
          )
          cc['CC_TREND_BALANCE_' + str(window) + 'M_MINUS_OVERALL'] = (
              cc['CC_WINDOW_' + str(window) + '_BALANCE_MEAN'] - cc['CC_OVERALL_BALANCE_MEAN']
          )
      if 'CC_WINDOW_' + str(window) + '_PAYMENT_MEAN' in cc.columns:
          cc['CC_TREND_PAYMENT_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
              cc['CC_WINDOW_' + str(window) + '_PAYMENT_MEAN'], cc['CC_OVERALL_PAYMENT_MEAN']
          )
          cc['CC_TREND_PAYMENT_' + str(window) + 'M_MINUS_OVERALL'] = (
              cc['CC_WINDOW_' + str(window) + '_PAYMENT_MEAN'] - cc['CC_OVERALL_PAYMENT_MEAN']
          )
      if 'CC_WINDOW_' + str(window) + '_DRAWINGS_MEAN' in cc.columns:
          cc['CC_TREND_DRAWINGS_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
              cc['CC_WINDOW_' + str(window) + '_DRAWINGS_MEAN'], cc['CC_OVERALL_DRAWINGS_MEAN']
          )

  cc = astypeFloat32(cc)
  clean()
  return cc

def ccRatioFeatures(cc_features):
  cc = cc_features

  cc['CC_RATIO_UTILIZATION_OVERALL'] = safeDivide(
    cc['CC_OVERALL_BALANCE_MEAN'],
    cc['CC_OVERALL_CREDIT_LIMIT_MEAN']
  )

  cc['CC_RATIO_PAYMENT_BALANCE_OVERALL'] = safeDivide(
    cc['CC_OVERALL_PAYMENT_MEAN'],
    cc['CC_OVERALL_BALANCE_MEAN']
  )

  cc['CC_RATIO_DRAWINGS_LIMIT_OVERALL'] = safeDivide(
    cc['CC_OVERALL_DRAWINGS_MEAN'],
    cc['CC_OVERALL_CREDIT_LIMIT_MEAN']
  )

  cc['CC_RATIO_RECEIVABLE_LIMIT_OVERALL'] = safeDivide(
    cc['CC_OVERALL_RECEIVABLE_MEAN'],
    cc['CC_OVERALL_CREDIT_LIMIT_MEAN']
  )

  for window in [1, 3, 6, 12, 24]:
    prefix = 'CC_WINDOW_' + str(window) + '_'

    if prefix + 'BALANCE_MEAN' in cc.columns:
      cc['CC_RATIO_UTILIZATION_' + str(window) + 'M'] = safeDivide(
        cc[prefix + 'BALANCE_MEAN'],
        cc[prefix + 'CREDIT_LIMIT_MEAN']
      )

      cc['CC_RATIO_PAYMENT_BALANCE_' + str(window) + 'M'] = safeDivide(
        cc[prefix + 'PAYMENT_MEAN'],
        cc[prefix + 'BALANCE_MEAN']
      )

      cc['CC_RATIO_DRAWINGS_LIMIT_' + str(window) + 'M'] = safeDivide(
        cc[prefix + 'DRAWINGS_MEAN'],
        cc[prefix + 'CREDIT_LIMIT_MEAN']
      )

      cc['CC_RATIO_RECEIVABLE_LIMIT_' + str(window) + 'M'] = safeDivide(
        cc[prefix + 'RECEIVABLE_MEAN'],
        cc[prefix + 'CREDIT_LIMIT_MEAN']
      )

  cc = astypeFloat32(cc)
  clean()

  return cc

def ccDelinquencyFeatures(cc_features):
  cc = cc_features

  cc['CC_DELINQ_DPD_SEVERITY_RATIO'] = safeDivide(
    cc['CC_OVERALL_DPD_DEF_SUM'],
    cc['CC_OVERALL_DPD_SUM']
  )

  cc['CC_DELINQ_HAS_DPD'] = (
    cc['CC_OVERALL_DPD_MAX'] > 0
  ).astype(np.float32)

  cc['CC_DELINQ_HAS_DPD_DEF'] = (
    cc['CC_OVERALL_DPD_DEF_MAX'] > 0
  ).astype(np.float32)

  for window in [1, 3, 6, 12, 24]:
    prefix = 'CC_WINDOW_' + str(window) + '_'

    if prefix + 'DPD_MEAN' in cc.columns:
      cc['CC_DELINQ_DPD_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
        cc[prefix + 'DPD_MEAN'],
        cc['CC_OVERALL_DPD_MEAN']
      )

      cc['CC_DELINQ_DPD_DEF_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
        cc[prefix + 'DPD_DEF_MEAN'],
        cc['CC_OVERALL_DPD_DEF_MEAN']
      )

  cc = astypeFloat32(cc)
  clean()

  return cc

In [ ]:
POS_CASH_balance = pd.read_csv(DATA_DIR / "POS_CASH_balance.csv")

def buildPOSFeatures(POS_CASH_balance):
  pos = POS_CASH_balance

  pos_overall = pos.groupby('SK_ID_CURR').agg(
    NUM_PREV_ACCOUNTS=('SK_ID_PREV', 'nunique'),
    RECORD_COUNT=('SK_ID_PREV', 'count'),

    CNT_INSTALMENT_MEAN=('CNT_INSTALMENT', 'mean'),
    CNT_INSTALMENT_MAX=('CNT_INSTALMENT', 'max'),
    CNT_INSTALMENT_SUM=('CNT_INSTALMENT', 'sum'),

    CNT_INSTALMENT_FUTURE_MEAN=('CNT_INSTALMENT_FUTURE', 'mean'),
    CNT_INSTALMENT_FUTURE_MAX=('CNT_INSTALMENT_FUTURE', 'max'),
    CNT_INSTALMENT_FUTURE_SUM=('CNT_INSTALMENT_FUTURE', 'sum'),

    DPD_MEAN=('SK_DPD', 'mean'),
    DPD_MAX=('SK_DPD', 'max'),
    DPD_SUM=('SK_DPD', 'sum'),

    DPD_DEF_MEAN=('SK_DPD_DEF', 'mean'),
    DPD_DEF_MAX=('SK_DPD_DEF', 'max'),
    DPD_DEF_SUM=('SK_DPD_DEF', 'sum')
  ).reset_index()

  cols = []
  for col in pos_overall.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('POS_OVERALL_' + col)

  pos_overall.columns = cols
  pos_overall = astypeFloat32(pos_overall)

  del pos
  clean()

  return pos_overall

def buildPOSWindowFeatures(POS_CASH_balance):
  pos = POS_CASH_balance

  windows = [1,3,6,12,24]

  pos_windows = None

  for window in windows:
    temp = pos[pos['MONTHS_BALANCE']>= -window]
    temp = temp.groupby('SK_ID_CURR').agg(
        CNT_INSTALMENT_MEAN=('CNT_INSTALMENT', 'mean'),
        CNT_INSTALMENT_SUM=('CNT_INSTALMENT', 'sum'),

        CNT_INSTALMENT_FUTURE_MEAN=('CNT_INSTALMENT_FUTURE', 'mean'),
        CNT_INSTALMENT_FUTURE_SUM=('CNT_INSTALMENT_FUTURE', 'sum'),

        DPD_MEAN=('SK_DPD', 'mean'),
        DPD_MAX=('SK_DPD', 'max'),

        DPD_DEF_MEAN=('SK_DPD_DEF', 'mean'),
        DPD_DEF_MAX=('SK_DPD_DEF', 'max'),

        ACTIVE_MONTHS=('MONTHS_BALANCE', 'count')
    ).reset_index()

    cols = []

    for col in temp.columns:
      if col == 'SK_ID_CURR':
        cols.append(col)
      else:
        cols.append('POS_WINDOW_' + str(window) + '_' + col)

    temp.columns = cols
    temp = astypeFloat32(temp)

    if pos_windows is None:
      pos_windows = temp
    else:
      pos_windows = pos_windows.merge(temp, on='SK_ID_CURR', how='left')

    clean()

  del pos
  clean()

  return pos_windows

def buildPOSStatusFeatures(POS_CASH_balance):
  pos = POS_CASH_balance[['SK_ID_CURR','MONTHS_BALANCE','NAME_CONTRACT_STATUS']].copy()

  status = pd.get_dummies(pos[['SK_ID_CURR','NAME_CONTRACT_STATUS']],columns=['NAME_CONTRACT_STATUS'])

  status = status.groupby('SK_ID_CURR').agg('sum').reset_index()

  status_cols = []
  for col in status.columns:
    if col == 'SK_ID_CURR':
      status_cols.append(col)
    else:
      status_cols.append('POS_STATUS_' + col.replace('NAME_CONTRACT_STATUS_',''))

  status.columns = status_cols

  total = status.drop(columns=['SK_ID_CURR']).sum(axis=1)

  for col in status.columns:
    if col!='SK_ID_CURR':
      ratio_name = col + '_RATIO'
      status[ratio_name] = safeDivide(status[col],total)

  for window in [3,6,12]:
    temp = pos[pos['MONTHS_BALANCE']>= -window]

    temp = pd.get_dummies(temp[['SK_ID_CURR','NAME_CONTRACT_STATUS']],columns=['NAME_CONTRACT_STATUS'])
    temp = temp.groupby('SK_ID_CURR').agg('sum').reset_index()

    cols=[]
    for col in temp.columns:
      if col == 'SK_ID_CURR':
        cols.append(col)
      else:
        cols.append('POS_STATUS_WINDOW_' + str(window) + '_'+col.replace('NAME_CONTRACT_STATUS',''))
    temp.columns = cols

    total = temp.drop(columns=['SK_ID_CURR']).sum(axis=1)
    for col in temp.columns:
      if col != 'SK_ID_CURR':
        ratio_name = col + '_RATIO'
        temp[ratio_name] = safeDivide(temp[col], total)

    status = status.merge(temp, on='SK_ID_CURR', how='left')

  status = astypeFloat32(status)

  del pos, temp
  clean()

  return status

def posTrendFeatures(pos_features):
  pos = pos_features

  for window in [1, 3, 6, 12, 24]:
    prefix = 'POS_WINDOW_' + str(window) + '_'

    if prefix + 'CNT_INSTALMENT_FUTURE_MEAN' in pos.columns:
      pos['POS_TREND_FUTURE_INSTALMENTS_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
        pos[prefix + 'CNT_INSTALMENT_FUTURE_MEAN'],
        pos['POS_OVERALL_CNT_INSTALMENT_FUTURE_MEAN']
      )

      pos['POS_TREND_DPD_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
        pos[prefix + 'DPD_MEAN'],
        pos['POS_OVERALL_DPD_MEAN']
      )

      pos['POS_TREND_INSTALMENTS_' + str(window) + 'M_OVERALL_RATIO'] = safeDivide(
        pos[prefix + 'CNT_INSTALMENT_MEAN'],
        pos['POS_OVERALL_CNT_INSTALMENT_MEAN']
      )

  pos = astypeFloat32(pos)
  clean()

  return pos

In [ ]:
installments_payments = pd.read_csv(DATA_DIR / "installments_payments.csv")

def prepareInstallments(installments_payments):
  inst = installments_payments.copy()
  inst['PAYMENT_DIFF'] = inst['AMT_PAYMENT'] - inst['AMT_INSTALMENT']
  inst['PAYMENT_RATIO'] = safeDivide(inst['AMT_PAYMENT'], inst['AMT_INSTALMENT'])
  inst['DPD'] = np.maximum(0,((inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT'])))
  inst['DBD'] = np.maximum(0,((inst['DAYS_INSTALMENT'] - inst['DAYS_ENTRY_PAYMENT'])))
  return inst

def buildInstallmentFeatures(installments_payments):
  inst = prepareInstallments(installments_payments)
  inst_overall = inst.groupby('SK_ID_CURR').agg(
    NUM_PREV_ACCOUNTS=('SK_ID_PREV', 'nunique'),
    NUM_INSTALMENT_VERSION_MEAN=('NUM_INSTALMENT_VERSION', 'mean'),
    NUM_INSTALMENT_VERSION_MAX=('NUM_INSTALMENT_VERSION', 'max'),
    NUM_INSTALMENT_NUMBER_MEAN=('NUM_INSTALMENT_NUMBER', 'mean'),
    NUM_INSTALMENT_NUMBER_MAX=('NUM_INSTALMENT_NUMBER', 'max'),
    AMT_INSTALMENT_MEAN=('AMT_INSTALMENT', 'mean'),
    AMT_INSTALMENT_MAX=('AMT_INSTALMENT', 'max'),
    AMT_INSTALMENT_SUM=('AMT_INSTALMENT', 'sum'),
    AMT_PAYMENT_MEAN=('AMT_PAYMENT', 'mean'),
    AMT_PAYMENT_MAX=('AMT_PAYMENT', 'max'),
    AMT_PAYMENT_SUM=('AMT_PAYMENT', 'sum'),
    PAYMENT_DIFF_MEAN=('PAYMENT_DIFF', 'mean'),
    PAYMENT_DIFF_MIN=('PAYMENT_DIFF', 'min'),
    PAYMENT_DIFF_MAX=('PAYMENT_DIFF', 'max'),
    PAYMENT_DIFF_SUM=('PAYMENT_DIFF', 'sum'),
    PAYMENT_RATIO_MEAN=('PAYMENT_RATIO', 'mean'),
    PAYMENT_RATIO_MIN=('PAYMENT_RATIO', 'min'),
    PAYMENT_RATIO_MAX=('PAYMENT_RATIO', 'max'),
    DPD_MEAN=('DPD', 'mean'),
    DPD_MAX=('DPD', 'max'),
    DPD_SUM=('DPD', 'sum'),
    DBD_MEAN=('DBD', 'mean'),
    DBD_MAX=('DBD', 'max'),
    DBD_SUM=('DBD', 'sum'),
    RECORD_COUNT=('SK_ID_PREV', 'count')
  ).reset_index()

  cols = []
  for col in inst_overall.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('INST_OVERALL_' + col)
  inst_overall.columns = cols
  inst_overall = astypeFloat32(inst_overall)

  del inst
  clean()
  return inst_overall


def buildInstallmentWindowFeatures(installments_payments):
  inst = prepareInstallments(installments_payments)
  windows = [30, 60, 90, 180, 365]
  inst_windows = None

  for window in windows:
    temp = inst[inst['DAYS_INSTALMENT'] >= -window]
    temp = temp.groupby('SK_ID_CURR').agg(
      AMT_PAYMENT_MEAN=('AMT_PAYMENT', 'mean'),
      AMT_PAYMENT_SUM=('AMT_PAYMENT', 'sum'),
      AMT_INSTALMENT_MEAN=('AMT_INSTALMENT', 'mean'),
      AMT_INSTALMENT_SUM=('AMT_INSTALMENT', 'sum'),
      PAYMENT_DIFF_MEAN=('PAYMENT_DIFF', 'mean'),
      PAYMENT_DIFF_SUM=('PAYMENT_DIFF', 'sum'),
      PAYMENT_RATIO_MEAN=('PAYMENT_RATIO', 'mean'),
      DPD_MEAN=('DPD', 'mean'),
      DPD_MAX=('DPD', 'max'),
      DPD_SUM=('DPD', 'sum'),
      DBD_MEAN=('DBD', 'mean'),
      DBD_MAX=('DBD', 'max'),
      ACTIVE_INSTALLMENTS=('SK_ID_PREV', 'count')
    ).reset_index()

    cols = []
    for col in temp.columns:
      if col == 'SK_ID_CURR':
        cols.append(col)
      else:
        cols.append('INST_WINDOW_' + str(window) + 'D_' + col)
    temp.columns = cols
    temp = astypeFloat32(temp)

    if inst_windows is None:
      inst_windows = temp
    else:
      inst_windows = inst_windows.merge(temp, on='SK_ID_CURR', how='left')

  del inst, temp
  clean()
  return inst_windows


def installmentTrendFeatures(inst_features):
  inst = inst_features
  for window in [30, 90, 365]:
    prefix = 'INST_WINDOW_' + str(window) + 'D_'
    if prefix + 'AMT_PAYMENT_MEAN' in inst.columns:
      inst['INST_TREND_PAYMENT_' + str(window) + 'D_OVERALL_RATIO'] = safeDivide(
        inst[prefix + 'AMT_PAYMENT_MEAN'], inst['INST_OVERALL_AMT_PAYMENT_MEAN']
      )
      inst['INST_TREND_PAYMENT_' + str(window) + 'D_MINUS_OVERALL'] = (
        inst[prefix + 'AMT_PAYMENT_MEAN'] - inst['INST_OVERALL_AMT_PAYMENT_MEAN']
      )
      inst['INST_TREND_DPD_' + str(window) + 'D_OVERALL_RATIO'] = safeDivide(inst[prefix + 'DPD_MEAN'], inst['INST_OVERALL_DPD_MEAN'])
      inst['INST_TREND_DPD_' + str(window) + 'D_MINUS_OVERALL'] = inst[prefix + 'DPD_MEAN'] - inst['INST_OVERALL_DPD_MEAN']
  inst = astypeFloat32(inst)
  clean()
  return inst

In [ ]:
def buildPreviousFeatures(previous_application):
  prev = previous_application.copy()
  date_cols = ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']
  for col in date_cols:
    prev[col] = prev[col].replace(365243, np.nan)

  prev['CREDIT_APPLICATION_RATIO'] = safeDivide(prev['AMT_CREDIT'], prev['AMT_APPLICATION'])
  prev['ANNUITY_CREDIT_RATIO'] = safeDivide(prev['AMT_ANNUITY'], prev['AMT_CREDIT'])
  prev['DOWN_PAYMENT_RATIO'] = safeDivide(prev['AMT_DOWN_PAYMENT'], prev['AMT_CREDIT'])
  prev['GOODS_CREDIT_RATIO'] = safeDivide(prev['AMT_GOODS_PRICE'], prev['AMT_CREDIT'])

  prev_overall = prev.groupby('SK_ID_CURR').agg(
    NUM_PREV_ACCOUNTS=('SK_ID_PREV', 'nunique'),
    APPLICATION_COUNT=('SK_ID_PREV', 'count'),
    AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),
    AMT_ANNUITY_MAX=('AMT_ANNUITY', 'max'),
    AMT_ANNUITY_SUM=('AMT_ANNUITY', 'sum'),
    AMT_APPLICATION_MEAN=('AMT_APPLICATION', 'mean'),
    AMT_APPLICATION_MAX=('AMT_APPLICATION', 'max'),
    AMT_APPLICATION_SUM=('AMT_APPLICATION', 'sum'),
    AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
    AMT_CREDIT_MAX=('AMT_CREDIT', 'max'),
    AMT_CREDIT_SUM=('AMT_CREDIT', 'sum'),
    AMT_DOWN_PAYMENT_MEAN=('AMT_DOWN_PAYMENT', 'mean'),
    AMT_DOWN_PAYMENT_MAX=('AMT_DOWN_PAYMENT', 'max'),
    AMT_DOWN_PAYMENT_SUM=('AMT_DOWN_PAYMENT', 'sum'),
    AMT_GOODS_PRICE_MEAN=('AMT_GOODS_PRICE', 'mean'),
    AMT_GOODS_PRICE_MAX=('AMT_GOODS_PRICE', 'max'),
    AMT_GOODS_PRICE_SUM=('AMT_GOODS_PRICE', 'sum'),
    HOUR_APPR_PROCESS_START_MEAN=('HOUR_APPR_PROCESS_START', 'mean'),
    RATE_DOWN_PAYMENT_MEAN=('RATE_DOWN_PAYMENT', 'mean'),
    RATE_DOWN_PAYMENT_MAX=('RATE_DOWN_PAYMENT', 'max'),
    DAYS_DECISION_MIN=('DAYS_DECISION', 'min'),
    DAYS_DECISION_MAX=('DAYS_DECISION', 'max'),
    DAYS_DECISION_MEAN=('DAYS_DECISION', 'mean'),
    CNT_PAYMENT_MEAN=('CNT_PAYMENT', 'mean'),
    CNT_PAYMENT_MAX=('CNT_PAYMENT', 'max'),
    CNT_PAYMENT_SUM=('CNT_PAYMENT', 'sum'),
    CREDIT_APPLICATION_RATIO_MEAN=('CREDIT_APPLICATION_RATIO', 'mean'),
    CREDIT_APPLICATION_RATIO_MAX=('CREDIT_APPLICATION_RATIO', 'max'),
    ANNUITY_CREDIT_RATIO_MEAN=('ANNUITY_CREDIT_RATIO', 'mean'),
    ANNUITY_CREDIT_RATIO_MAX=('ANNUITY_CREDIT_RATIO', 'max'),
    DOWN_PAYMENT_RATIO_MEAN=('DOWN_PAYMENT_RATIO', 'mean'),
    DOWN_PAYMENT_RATIO_MAX=('DOWN_PAYMENT_RATIO', 'max'),
    GOODS_CREDIT_RATIO_MEAN=('GOODS_CREDIT_RATIO', 'mean'),
    GOODS_CREDIT_RATIO_MAX=('GOODS_CREDIT_RATIO', 'max')
  ).reset_index()

  cols = []
  for col in prev_overall.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('PREV_OVERALL_' + col)
  prev_overall.columns = cols
  prev_overall = astypeFloat32(prev_overall)

  del prev, date_cols
  clean()
  return prev_overall


def buildPreviousStatusFeatures(previous_application):
  prev = previous_application[['SK_ID_CURR','NAME_CONTRACT_STATUS','NAME_CONTRACT_TYPE','PRODUCT_COMBINATION']].copy()
  prev = pd.get_dummies(prev, columns=['NAME_CONTRACT_STATUS','NAME_CONTRACT_TYPE','PRODUCT_COMBINATION'])
  prev = prev.groupby('SK_ID_CURR').sum().reset_index()

  cols = []
  for col in prev.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('PREV_STATUS_' + col)
  prev.columns = cols
  prev = astypeFloat32(prev)
  clean()
  return prev


def buildPreviousWindowFeatures(previous_application):
  prev = previous_application.copy()
  windows = [30, 90, 180, 365, 730]
  prev_windows = None

  for window in windows:
    temp = prev[prev['DAYS_DECISION'] >= -window]
    temp = temp.groupby('SK_ID_CURR').agg(
      APPLICATION_COUNT=('SK_ID_PREV', 'count'),
      AMT_APPLICATION_MEAN=('AMT_APPLICATION', 'mean'),
      AMT_APPLICATION_SUM=('AMT_APPLICATION', 'sum'),
      AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
      AMT_CREDIT_SUM=('AMT_CREDIT', 'sum'),
      AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),
      CNT_PAYMENT_MEAN=('CNT_PAYMENT', 'mean'),
      DAYS_DECISION_MEAN=('DAYS_DECISION', 'mean'),
      DAYS_DECISION_MAX=('DAYS_DECISION', 'max')
    ).reset_index()

    cols = []
    for col in temp.columns:
      if col == 'SK_ID_CURR':
        cols.append(col)
      else:
        cols.append('PREV_WINDOW_' + str(window) + 'D_' + col)
    temp.columns = cols
    temp = astypeFloat32(temp)

    if prev_windows is None:
      prev_windows = temp
    else:
      prev_windows = prev_windows.merge(temp, on='SK_ID_CURR', how='left')

  del prev, temp
  clean()
  return prev_windows


def previousTrendFeatures(prev_features):
  prev = prev_features
  for window in [30, 90, 180, 365, 730]:
    prefix = 'PREV_WINDOW_' + str(window) + 'D_'
    if prefix + 'AMT_APPLICATION_MEAN' in prev.columns:
      prev['PREV_TREND_APPLICATION_' + str(window) + 'D_OVERALL_RATIO'] = safeDivide(
        prev[prefix + 'AMT_APPLICATION_MEAN'], prev['PREV_OVERALL_AMT_APPLICATION_MEAN']
      )
      prev['PREV_TREND_CREDIT_' + str(window) + 'D_OVERALL_RATIO'] = safeDivide(
        prev[prefix + 'AMT_CREDIT_MEAN'], prev['PREV_OVERALL_AMT_CREDIT_MEAN']
      )
      prev['PREV_TREND_APPLICATION_COUNT_' + str(window) + 'D_RATIO'] = safeDivide(
        prev[prefix + 'APPLICATION_COUNT'], prev['PREV_OVERALL_APPLICATION_COUNT']
      )
  prev = astypeFloat32(prev)
  clean()
  return prev

In [ ]:
def buildBureauFeatures(bureau):
  bur = bureau.copy()
  bur['DEBT_CREDIT_RATIO'] = safeDivide(bur['AMT_CREDIT_SUM_DEBT'], bur['AMT_CREDIT_SUM'])
  bur['OVERDUE_CREDIT_RATIO'] = safeDivide(bur['AMT_CREDIT_SUM_OVERDUE'], bur['AMT_CREDIT_SUM'])

  bur_overall = bur.groupby('SK_ID_CURR').agg(
    NUM_BUREAU_RECORDS=('SK_ID_BUREAU', 'nunique'),
    RECORD_COUNT=('SK_ID_BUREAU', 'count'),
    DAYS_CREDIT_MIN=('DAYS_CREDIT', 'min'),
    DAYS_CREDIT_MAX=('DAYS_CREDIT', 'max'),
    DAYS_CREDIT_MEAN=('DAYS_CREDIT', 'mean'),
    CREDIT_DAY_OVERDUE_MAX=('CREDIT_DAY_OVERDUE', 'max'),
    CREDIT_DAY_OVERDUE_MEAN=('CREDIT_DAY_OVERDUE', 'mean'),
    CREDIT_DAY_OVERDUE_SUM=('CREDIT_DAY_OVERDUE', 'sum'),
    DAYS_CREDIT_ENDDATE_MIN=('DAYS_CREDIT_ENDDATE', 'min'),
    DAYS_CREDIT_ENDDATE_MAX=('DAYS_CREDIT_ENDDATE', 'max'),
    DAYS_CREDIT_ENDDATE_MEAN=('DAYS_CREDIT_ENDDATE', 'mean'),
    DAYS_ENDDATE_FACT_MIN=('DAYS_ENDDATE_FACT', 'min'),
    DAYS_ENDDATE_FACT_MAX=('DAYS_ENDDATE_FACT', 'max'),
    DAYS_ENDDATE_FACT_MEAN=('DAYS_ENDDATE_FACT', 'mean'),
    AMT_CREDIT_MAX_OVERDUE_MEAN=('AMT_CREDIT_MAX_OVERDUE', 'mean'),
    AMT_CREDIT_MAX_OVERDUE_MAX=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    CNT_CREDIT_PROLONG_SUM=('CNT_CREDIT_PROLONG', 'sum'),
    CNT_CREDIT_PROLONG_MAX=('CNT_CREDIT_PROLONG', 'max'),
    AMT_CREDIT_SUM_MEAN=('AMT_CREDIT_SUM', 'mean'),
    AMT_CREDIT_SUM_MAX=('AMT_CREDIT_SUM', 'max'),
    AMT_CREDIT_SUM_SUM=('AMT_CREDIT_SUM', 'sum'),
    AMT_CREDIT_SUM_DEBT_MEAN=('AMT_CREDIT_SUM_DEBT', 'mean'),
    AMT_CREDIT_SUM_DEBT_MAX=('AMT_CREDIT_SUM_DEBT', 'max'),
    AMT_CREDIT_SUM_DEBT_SUM=('AMT_CREDIT_SUM_DEBT', 'sum'),
    AMT_CREDIT_SUM_LIMIT_MEAN=('AMT_CREDIT_SUM_LIMIT', 'mean'),
    AMT_CREDIT_SUM_LIMIT_MAX=('AMT_CREDIT_SUM_LIMIT', 'max'),
    AMT_CREDIT_SUM_LIMIT_SUM=('AMT_CREDIT_SUM_LIMIT', 'sum'),
    AMT_CREDIT_SUM_OVERDUE_MEAN=('AMT_CREDIT_SUM_OVERDUE', 'mean'),
    AMT_CREDIT_SUM_OVERDUE_MAX=('AMT_CREDIT_SUM_OVERDUE', 'max'),
    AMT_CREDIT_SUM_OVERDUE_SUM=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    DAYS_CREDIT_UPDATE_MIN=('DAYS_CREDIT_UPDATE', 'min'),
    DAYS_CREDIT_UPDATE_MAX=('DAYS_CREDIT_UPDATE', 'max'),
    DAYS_CREDIT_UPDATE_MEAN=('DAYS_CREDIT_UPDATE', 'mean'),
    AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),
    AMT_ANNUITY_MAX=('AMT_ANNUITY', 'max'),
    AMT_ANNUITY_SUM=('AMT_ANNUITY', 'sum'),
    DEBT_CREDIT_RATIO_MEAN=('DEBT_CREDIT_RATIO', 'mean'),
    DEBT_CREDIT_RATIO_MAX=('DEBT_CREDIT_RATIO', 'max'),
    OVERDUE_CREDIT_RATIO_MEAN=('OVERDUE_CREDIT_RATIO', 'mean'),
    OVERDUE_CREDIT_RATIO_MAX=('OVERDUE_CREDIT_RATIO', 'max')
  ).reset_index()

  cols = []
  for col in bur_overall.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('BUR_OVERALL_' + col)
  bur_overall.columns = cols
  bur_overall = astypeFloat32(bur_overall)

  del bur
  clean()
  return bur_overall


def buildBureauStatusFeatures(bureau):
  cols_to_encode = ['CREDIT_ACTIVE', 'CREDIT_TYPE', 'CREDIT_CURRENCY']
  bur = bureau[['SK_ID_CURR'] + cols_to_encode].copy()
  bur = pd.get_dummies(bur, columns=cols_to_encode)
  bur = bur.groupby('SK_ID_CURR').agg('sum').reset_index()

  cols = []
  for col in bur.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('BUR_STATUS_' + col)
  bur.columns = cols
  bur = astypeFloat32(bur)
  clean()
  return bur


def buildBureauWindowFeatures(bureau):
  bur = bureau.copy()
  bur['DEBT_CREDIT_RATIO'] = safeDivide(bur['AMT_CREDIT_SUM_DEBT'], bur['AMT_CREDIT_SUM'])
  windows = [180, 365, 730, 1095]
  bur_windows = None

  for window in windows:
    temp = bur[bur['DAYS_CREDIT'] >= -window]
    temp = temp.groupby('SK_ID_CURR').agg(
      BUREAU_RECORD_COUNT=('SK_ID_BUREAU', 'count'),
      AMT_CREDIT_SUM_MEAN=('AMT_CREDIT_SUM', 'mean'),
      AMT_CREDIT_SUM_SUM=('AMT_CREDIT_SUM', 'sum'),
      AMT_CREDIT_SUM_DEBT_MEAN=('AMT_CREDIT_SUM_DEBT', 'mean'),
      AMT_CREDIT_SUM_DEBT_SUM=('AMT_CREDIT_SUM_DEBT', 'sum'),
      CREDIT_DAY_OVERDUE_MAX=('CREDIT_DAY_OVERDUE', 'max'),
      CREDIT_DAY_OVERDUE_MEAN=('CREDIT_DAY_OVERDUE', 'mean'),
      DEBT_CREDIT_RATIO_MEAN=('DEBT_CREDIT_RATIO', 'mean'),
      DEBT_CREDIT_RATIO_MAX=('DEBT_CREDIT_RATIO', 'max')
    ).reset_index()

    cols = []
    for col in temp.columns:
      if col == 'SK_ID_CURR':
        cols.append(col)
      else:
        cols.append('BUR_WINDOW_' + str(window) + 'D_' + col)
    temp.columns = cols
    temp = astypeFloat32(temp)

    if bur_windows is None:
      bur_windows = temp
    else:
      bur_windows = bur_windows.merge(temp, on='SK_ID_CURR', how='left')

  del bur, temp
  clean()
  return bur_windows


def buildBureauBalanceFeatures(bureau, bureau_balance):
  bur_map = bureau[['SK_ID_BUREAU', 'SK_ID_CURR']].copy()
  bb = bureau_balance.merge(bur_map, on='SK_ID_BUREAU', how='left')
  bb = bb.dropna(subset=['SK_ID_CURR'])
  bb['SK_ID_CURR'] = bb['SK_ID_CURR'].astype(np.int64)

  status = pd.get_dummies(bb[['SK_ID_CURR', 'STATUS']], columns=['STATUS'])
  status_sum = status.groupby('SK_ID_CURR').sum().reset_index()
  status_mean = status.groupby('SK_ID_CURR').mean().reset_index()

  cols = []
  for col in status_sum.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('BB_STATUS_SUM_' + col.replace('STATUS_', ''))
  status_sum.columns = cols

  cols = []
  for col in status_mean.columns:
    if col == 'SK_ID_CURR':
      cols.append(col)
    else:
      cols.append('BB_STATUS_MEAN_' + col.replace('STATUS_', ''))
  status_mean.columns = cols

  bb_features = status_sum.merge(status_mean, on='SK_ID_CURR', how='left')

  for window in [3, 6, 12, 24]:
    temp = bb[bb['MONTHS_BALANCE'] >= -window]
    temp = pd.get_dummies(temp[['SK_ID_CURR', 'STATUS']], columns=['STATUS'])
    temp = temp.groupby('SK_ID_CURR').sum().reset_index()
    cols = []
    for col in temp.columns:
      if col == 'SK_ID_CURR':
        cols.append(col)
      else:
        cols.append('BB_WINDOW_' + str(window) + '_' + col.replace('STATUS_', ''))
    temp.columns = cols
    bb_features = bb_features.merge(temp, on='SK_ID_CURR', how='left')

  bb_features = astypeFloat32(bb_features)
  del bur_map, bb, status, status_sum, status_mean, temp
  clean()
  return bb_features


def bureauTrendFeatures(bur_features):
  bur = bur_features
  for window in [180, 365, 730, 1095]:
    prefix = 'BUR_WINDOW_' + str(window) + 'D_'
    if prefix + 'DEBT_CREDIT_RATIO_MEAN' in bur.columns:
      bur['BUR_TREND_DEBT_CREDIT_' + str(window) + 'D_OVERALL_RATIO'] = safeDivide(
        bur[prefix + 'DEBT_CREDIT_RATIO_MEAN'], bur['BUR_OVERALL_DEBT_CREDIT_RATIO_MEAN']
      )
      bur['BUR_TREND_OVERDUE_' + str(window) + 'D_OVERALL_RATIO'] = safeDivide(
        bur[prefix + 'CREDIT_DAY_OVERDUE_MEAN'], bur['BUR_OVERALL_CREDIT_DAY_OVERDUE_MEAN']
      )
  bur = astypeFloat32(bur)
  clean()
  return bur

In [ ]:
def buildFinalTabularFeatures():
  application_train = pd.read_csv(DATA_DIR / "application_train.csv")
  master = buildApplicationFeatures(application_train)
  del application_train
  clean()

  credit_card_balance = pd.read_csv(DATA_DIR / "credit_card_balance.csv")
  cc = credit_card_balance
  cc_features = buildCCFeatures(cc)
  cc_features = cc_features.merge(buildCCWindowFeatures(cc), on='SK_ID_CURR', how='left')
  cc_features = ccTrendFeatures(cc_features)
  cc_features = ccRatioFeatures(cc_features)
  cc_features = ccDelinquencyFeatures(cc_features)
  master = mergeFeatureBlock(master, cc_features, 'credit card')
  del cc, cc_features
  clean()

  POS_CASH_balance = pd.read_csv(DATA_DIR / "POS_CASH_balance.csv")
  pos = POS_CASH_balance
  pos_features = buildPOSFeatures(pos)
  pos_features = pos_features.merge(buildPOSWindowFeatures(pos), on='SK_ID_CURR', how='left')
  pos_features = pos_features.merge(buildPOSStatusFeatures(pos), on='SK_ID_CURR', how='left')
  pos_features = posTrendFeatures(pos_features)
  master = mergeFeatureBlock(master, pos_features, 'POS')
  del pos, pos_features
  clean()

  installments_payments = pd.read_csv(DATA_DIR / "installments_payments.csv")
  inst = installments_payments
  inst_features = buildInstallmentFeatures(inst)
  inst_features = inst_features.merge(buildInstallmentWindowFeatures(inst), on='SK_ID_CURR', how='left')
  inst_features = installmentTrendFeatures(inst_features)
  master = mergeFeatureBlock(master, inst_features, 'installments')
  del inst, inst_features
  clean()

  previous_application = pd.read_csv(DATA_DIR / "previous_application.csv")
  prev = previous_application
  prev_features = buildPreviousFeatures(prev)
  prev_features = prev_features.merge(buildPreviousStatusFeatures(prev), on='SK_ID_CURR', how='left')
  prev_features = prev_features.merge(buildPreviousWindowFeatures(prev), on='SK_ID_CURR', how='left')
  prev_features = previousTrendFeatures(prev_features)
  master = mergeFeatureBlock(master, prev_features, 'previous applications')
  del prev, prev_features
  clean()

  bureau = pd.read_csv(DATA_DIR / "bureau.csv")
  bureau_balance = pd.read_csv(DATA_DIR / "bureau_balance.csv")
  bur_features = buildBureauFeatures(bureau)
  bur_features = bur_features.merge(buildBureauStatusFeatures(bureau), on='SK_ID_CURR', how='left')
  bur_features = bur_features.merge(buildBureauWindowFeatures(bureau), on='SK_ID_CURR', how='left')
  bur_features = bur_features.merge(buildBureauBalanceFeatures(bureau, bureau_balance), on='SK_ID_CURR', how='left')
  bur_features = bureauTrendFeatures(bur_features)
  master = mergeFeatureBlock(master, bur_features, 'bureau')
  del bureau, bureau_balance, bur_features
  clean()

  master = master.replace([np.inf, -np.inf], np.nan)
  print('Total NaNs before final fill:', master.isna().sum().sum())

  for col in master.columns:
    if col not in ['SK_ID_CURR', 'TARGET']:
      if master[col].isna().any():
        master[col] = master[col].fillna(0)

  master = astypeFloat32(master)
  print(master.shape)
  print(master['TARGET'].value_counts())
  print('Total NaNs:', master.isna().sum().sum())
  print('Duplicate customers:', master['SK_ID_CURR'].duplicated().sum())

  master.to_pickle(ARTIFACT_DIR / "home_credit_tabular_features.pkl")
  return master


master = buildFinalTabularFeatures()

# **Model Training - LightGBM**



In [ ]:
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, confusion_matrix, RocCurveDisplay
import matplotlib.pyplot as plt
import seaborn as sns

master = pd.read_pickle(ARTIFACT_DIR / "home_credit_tabular_features.pkl")

def evaluate_model(name, y_true, probs, threshold):

    preds = (probs >= threshold).astype(int)

    print(f"\n{name}")
    print("-"*40)
    print("Accuracy :", accuracy_score(y_true, preds))
    print("Precision:", precision_score(y_true, preds))
    print("Recall   :", recall_score(y_true, preds))
    print("F1 Score :", f1_score(y_true, preds))
    print("ROC AUC  :", roc_auc_score(y_true, probs))
    print()
    print(confusion_matrix(y_true, preds))

In [ ]:
X = master.drop(columns=["SK_ID_CURR", "TARGET"])
y = master["TARGET"]

X = X.replace([np.inf, -np.inf], np.nan)

X.columns = [
  re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
  for col in X.columns
]

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1111, random_state=42, stratify=y_train_val)

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

In [ ]:
params = {
  'boosting_type': 'gbdt',
  'objective': 'binary',
  'metric': 'auc',
  'learning_rate': 0.03,
  'num_leaves': 31,
  'feature_fraction': 0.8,
  'bagging_fraction':0.8,
  'bagging_freq': 1,
  'random_state':42,
  'verbose': -1,
  'max_depth':6,
  'scale_pos_weight':11.35,
  'min_data_in_leaf':30
}

model = lgb.train(
  params,
  train_data,
  num_boost_round=800,
  valid_sets=[val_data],
  callbacks=[lgb.early_stopping(stopping_rounds=10)]
)

lgb_val_probs = model.predict(X_val)
print('AUC:', roc_auc_score(y_val, lgb_val_probs))

In [ ]:
lgb_threshold = 0
lgb_best_f1 = 0

for threshold in np.arange(0,1.00,0.01):
  preds = (lgb_val_probs >= threshold).astype(int)
  f1 = f1_score(y_val, preds)

  if f1 > lgb_best_f1:
    lgb_best_f1 = f1
    lgb_threshold = threshold

print('LightGBM Best threshold:', lgb_threshold)
print('LightGBM Best F1:', lgb_best_f1)

In [ ]:
lgb_test_probs = model.predict(X_test)

evaluate_model(
    "LightGBM",
    y_test,
    lgb_test_probs,
    lgb_threshold
)

In [ ]:
RocCurveDisplay.from_predictions(y_test, lgb_test_probs)
plt.title("LightGBM ROC Curve")
plt.show()

# **Model Training - XGBoost**

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

xgb_model = XGBClassifier(
    n_estimators = 1000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    scale_pos_weight=11.35,
    subsample=0.8,
    colsample_bytree=0.8,
    objective = 'binary:logistic',
    eval_metric = ['auc','logloss'],
    random_state=42,
    gamma=0.2,
    n_jobs=1,
    early_stopping_rounds = 10
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50)

xgb_val_probs = xgb_model.predict_proba(X_val)[:,1]
xgb_auc = roc_auc_score(y_val, xgb_val_probs)

print("XGBoost AUC:", xgb_auc)

In [ ]:
xgb_threshold = 0
xgb_best_f1 = 0

for threshold in np.arange(0, 1.01, 0.01):
    preds = (xgb_val_probs >= threshold).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > xgb_best_f1:
        xgb_best_f1 = f1
        xgb_threshold = threshold

print("XGBoost Best threshold:", xgb_threshold)
print("XGBoost Best validation F1:", xgb_best_f1)

In [ ]:
xgb_test_probs = xgb_model.predict_proba(X_test)[:,1]

evaluate_model(
    "XGBoost",
    y_test,
    xgb_test_probs,
    xgb_threshold
)

In [ ]:
RocCurveDisplay.from_predictions(y_test, xgb_test_probs)
plt.title("XGBoost ROC Curve")
plt.show()

# **Model Training - CatBoost**

In [ ]:
from catboost import CatBoostClassifier

cat_model=CatBoostClassifier(
    iterations=4000,
    learning_rate=0.03,
    depth=4,
    eval_metric='AUC',
    verbose=500,
    colsample_bylevel=0.8,
    random_state=42,
    objective='Logloss'
)

cat_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    use_best_model = True,
    early_stopping_rounds =50
)

In [ ]:
cat_val_probs = cat_model.predict_proba(X_val)[:,1]

cat_threshold = 0
cat_best_f1 = 0

for threshold in np.arange(0,1.01,0.01):
    preds = (cat_val_probs >= threshold).astype(int)
    f1 = f1_score(y_val,preds)

    if f1 > cat_best_f1:
        cat_best_f1 = f1
        cat_threshold = threshold

print("CatBoost Best threshold:", cat_threshold)
print("CatBoost Best validation F1:", cat_best_f1)

In [ ]:
cat_test_probs = cat_model.predict_proba(X_test)[:,1]

evaluate_model(
    "CatBoost",
    y_test,
    cat_test_probs,
    cat_threshold
)

In [ ]:
RocCurveDisplay.from_predictions(y_test, cat_test_probs)
plt.title("CatBoost ROC Curve")
plt.show()

# **Model Training - Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=500,
    class_weight = 'balanced_subsample',
    max_features = 0.05,
    min_samples_leaf = 50,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(
    X_train,
    y_train,
)

rf_val_probs = model_rf.predict_proba(X_val)[:,1]
rf_auc = roc_auc_score(y_val, rf_val_probs)

print("Random Forest AUC:", rf_auc)

In [ ]:
rf_threshold = 0
rf_best_f1 = 0

for threshold in np.arange(0, 1.01, 0.01):
    preds = (rf_val_probs >= threshold).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > rf_best_f1:
        rf_best_f1 = f1
        rf_threshold = threshold

print("Random Forest Best threshold:", rf_threshold)
print("Random Forest Best validation F1:", rf_best_f1)

In [ ]:
rf_test_probs = model_rf.predict_proba(X_test)[:, 1]

evaluate_model(
    "Random Forest",
    y_test,
    rf_test_probs,
    rf_threshold
)

In [ ]:
RocCurveDisplay.from_predictions(y_test, model_rf.predict_proba(X_test)[:,1])
plt.title("Random Forest ROC Curve")
plt.show()

# **Model Training - Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

lr_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

lr_model.fit(X_train, y_train)

lr_val_probs = lr_model.predict_proba(X_val)[:, 1]

In [ ]:
lr_threshold = 0
lr_best_f1 = 0

for threshold in np.arange(0, 1.01, 0.01):
    preds = (lr_val_probs >= threshold).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > lr_best_f1:
        lr_best_f1 = f1
        lr_threshold = threshold

print("Logistic Regression Best threshold:", lr_threshold)
print("Logistic Regression Best validation F1:", lr_best_f1)

In [ ]:
lr_test_probs = lr_model.predict_proba(X_test)[:, 1]

evaluate_model(
    "Logistic Regression",
    y_test,
    lr_test_probs,
    lr_threshold
)

In [ ]:
RocCurveDisplay.from_predictions(y_test, lr_model.predict_proba(X_test)[:,1])
plt.title("Logistic Regression ROC Curve")
plt.show()

# **ALL PLOTS**

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np
import os

model_preds = {
    "Logistic Regression": (y_test, lr_model.predict_proba(X_test)[:, 1]),
    "Random Forest": (y_test, model_rf.predict_proba(X_test)[:, 1]),
    "LightGBM": (y_test, model.predict(X_test)),
    "XGBoost": (y_test, xgb_model.predict_proba(X_test)[:, 1]),
    "CatBoost": (y_test, cat_model.predict_proba(X_test)[:, 1])
}

if os.path.exists(ARTIFACT_DIR / "rnn_probs.npy") and os.path.exists(ARTIFACT_DIR / "rnn_y_test.npy"):
    rnn_probs = np.load(ARTIFACT_DIR / "rnn_probs.npy")
    rnn_y_test = np.load(ARTIFACT_DIR / "rnn_y_test.npy")
    model_preds["LSTM"] = (rnn_y_test, rnn_probs)

plt.figure(figsize=(9, 7))

for name, (true_y, probs) in model_preds.items():
    fpr, tpr, _ = roc_curve(true_y, probs)
    auc_score = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{name} (AUC = {auc_score:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    color="black",
    label="Random Guess"
)

plt.title("ROC Curve Comparison Across Models")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.xlim(0, 1)
plt.ylim(0, 1.02)

plt.grid(alpha=0.3)
plt.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import os

rnn_probs_path = ARTIFACT_DIR / "rnn_probs.npy"
rnn_y_test_path = ARTIFACT_DIR / "rnn_y_test.npy"

model_outputs = {
    "Logistic Regression": (y_test, lr_test_probs, lr_threshold),
    "Random Forest": (y_test, rf_test_probs, rf_threshold),
    "LightGBM": (y_test, lgb_test_probs, lgb_threshold),
    "XGBoost": (y_test, xgb_test_probs, xgb_threshold),
    "CatBoost": (y_test, cat_test_probs, cat_threshold)
}

if os.path.exists(rnn_probs_path) and os.path.exists(rnn_y_test_path):
    rnn_probs = np.load(rnn_probs_path).reshape(-1)
    rnn_y_test = np.load(rnn_y_test_path).reshape(-1)

    print("RNN probabilities shape:", rnn_probs.shape)
    print("RNN labels shape:", rnn_y_test.shape)

    if len(rnn_probs) != len(rnn_y_test):
        raise ValueError(
            f"RNN probability and label lengths do not match: "
            f"{len(rnn_probs)} vs. {len(rnn_y_test)}"
        )

    model_outputs["LSTM"] = (
        rnn_y_test,
        rnn_probs,
        0.5
    )
else:
    raise FileNotFoundError(
        "Could not locate one or both LSTM files.\n"
        f"Probabilities found: {os.path.exists(rnn_probs_path)}\n"
        f"Labels found: {os.path.exists(rnn_y_test_path)}"
    )

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

all_cms = []

for name, (true_y, probs, threshold) in model_outputs.items():
    true_y = np.asarray(true_y).reshape(-1)
    probs = np.asarray(probs).reshape(-1)

    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(true_y, preds, labels=[0, 1])
    all_cms.append(cm)

vmax = max(cm.max() for cm in all_cms)

for ax, (name, (true_y, probs, threshold)) in zip(
    axes, model_outputs.items()
):
    true_y = np.asarray(true_y).reshape(-1)
    probs = np.asarray(probs).reshape(-1)

    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(true_y, preds, labels=[0, 1])

    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=vmax)

    ax.set_title(name, fontsize=12)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Non-default", "Default"])
    ax.set_yticklabels(["Non-default", "Default"])

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{cm[i, j]:,}",
                ha="center",
                va="center",
                fontsize=10,
                color="white" if cm[i, j] > vmax * 0.45 else "black"
            )

for ax in axes[len(model_outputs):]:
    ax.axis("off")

fig.suptitle(
    "Confusion Matrices for Evaluated Machine Learning Models",
    fontsize=16,
    y=0.98
)

fig.subplots_adjust(
    left=0.06,
    right=0.88,
    top=0.90,
    bottom=0.08,
    wspace=0.35,
    hspace=0.35
)

cbar_ax = fig.add_axes([0.90, 0.18, 0.02, 0.65])
fig.colorbar(im, cax=cbar_ax, label="Number of Applicants")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

master = pd.read_pickle(ARTIFACT_DIR / "home_credit_tabular_features.pkl")

def get_source(col):
    col = str(col).upper()
    if col.startswith("CC_"):
        return "Credit Card"
    elif col.startswith("POS_"):
        return "POS Cash"
    elif col.startswith("BUR_") or col.startswith("BB_"):
        return "Bureau"
    elif col.startswith("INST_"):
        return "Installments"
    elif col.startswith("PREV_"):
        return "Previous Applications"
    else:
        return "Application/Main"

feature_cols = [c for c in master.columns if c not in ["SK_ID_CURR", "TARGET"]]
source_counts = pd.Series([get_source(c) for c in feature_cols]).value_counts().sort_values()

plt.figure(figsize=(8, 5))
plt.barh(source_counts.index, source_counts.values)

plt.title("Feature Count by Data Source")
plt.xlabel("Number of Features")
plt.ylabel("Data Source")

plt.xlim(0, source_counts.max() + 20)

for i, value in enumerate(source_counts.values):
    plt.text(value + 2, i, str(value), va="center")

plt.tight_layout()
plt.show()

In [ ]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat_model.get_feature_importance()
})

importance_df = importance_df.sort_values("importance", ascending=False).head(20)
importance_df = importance_df.sort_values("importance")

plt.figure(figsize=(10, 8))
bars = plt.barh(importance_df["feature"], importance_df["importance"])

for bar in bars:
    width = bar.get_width()

    plt.text(
        width,
        bar.get_y() + bar.get_height()/2,
        f"{width:.1f}",
        va="center",
        fontsize=8
    )

plt.title("Top 20 CatBoost Feature Importances")
plt.xlabel("Importance Gain")
plt.ylabel("Feature")

locs, labels = plt.xticks()
plt.xticks(locs, rotation=90)

plt.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

master=pd.read_pickle(ARTIFACT_DIR / "home_credit_tabular_features.pkl")

numeric = master.select_dtypes(include=np.number)
numeric = numeric.drop(columns=["SK_ID_CURR"], errors="ignore")

corr = numeric.corr()
abs_corr = corr.abs()

upper = abs_corr.where(
    np.triu(np.ones(abs_corr.shape), k=1).astype(bool)
)

pairs = (
    upper.stack()
    .sort_values(ascending=False)
    .head(30)
)

top_features = list(
    set(feature for pair in pairs.index for feature in pair)
)

print(f"{len(top_features)} features selected")

corr_plot = corr.loc[top_features, top_features]

distance = 1 - np.abs(corr_plot)
np.fill_diagonal(distance.values, 0)
distance = distance.fillna(0) # Fill any remaining NaNs with 0
distance = np.maximum(0, distance) # Ensure all distances are non-negative

linkage_matrix = linkage(
    squareform(distance.values),
    method="average"
)

order = leaves_list(linkage_matrix)
corr_plot = corr_plot.iloc[order, order]

def shorten_feature_name(name):
    replacements = {
        "OVERALL": "OVR",
        "WINDOW": "WIN",
        "RECEIVABLE": "RECV",
        "UTILIZATION": "UTIL",
        "APPLICATION": "APP",
        "INSTALMENT": "INST",
        "CREDIT_LIMIT": "LIMIT",
        "DAYS_DECISION": "DECISION",
        "CONTRACT_STATUS": "STATUS",
        "NUMBER": "NUM",
        "AVERAGE": "AVG",
    }

    short_name = str(name)

    for old, new in replacements.items():
        short_name = short_name.replace(old, new)

    return short_name

short_labels = [
    shorten_feature_name(col)
    for col in corr_plot.columns
]

plt.figure(figsize=(12, 10))

im = plt.imshow(
    corr_plot,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    interpolation="nearest"
)

plt.xticks(
    range(len(short_labels)),
    short_labels,
    rotation=90,
    fontsize=8
)

plt.yticks(
    range(len(short_labels)),
    short_labels,
    fontsize=8
)

plt.colorbar(
    im,
    label="Pearson Correlation"
)

plt.title(
    "Correlation Structure of Highly Correlated Engineered Features"
)

plt.tight_layout()
plt.show()